# 3 — Prospect Agent (Researcher → Copywriter Pipeline)

Two specialist LLMs in sequence: a researcher gathers web facts about a prospect, then a copywriter uses those facts to draft a personalized outreach message.

**What you'll learn**
- Custom `TypedDict` state vs `MessagesState` — when typed fields beat a message list
- Two specialist agents in a linear pipeline: `researcher → copywriter`
- Passing structured state between nodes — how `research` in state flows from node 1 to node 2
- `DuckDuckGoSearchResults` as a retrieval tool called directly inside a node
- Role separation: one LLM gathers facts, a different LLM writes the message

**Companion examples:** 1-basic-local-rag (ReAct loop vs linear pipeline), 6-multi-agent-graph (three specialist agents)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-community duckduckgo-search python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Define state and data models ───────────────────────────────────────────
# Unlike MessagesState, a custom TypedDict lets you name each field explicitly.
# The researcher fills 'research', the copywriter reads it to fill 'output'.
# Neither node needs to know about the other's internals.
from typing import TypedDict

from pydantic import BaseModel, Field


class ProspectMetadata(BaseModel):
    first_name: str
    last_name: str
    company: str
    position: str


class ResearchOutput(BaseModel):
    search_query: str
    search_results: str


class OutreachOutput(BaseModel):
    generated_message: str = Field(description="Personalized outreach message (<=300 chars)")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")


class AgentState(TypedDict):
    prospect: ProspectMetadata
    research: ResearchOutput | None    # filled by researcher node
    output: OutreachOutput | None      # filled by copywriter node


print("State: prospect -> [researcher fills research] -> [copywriter fills output]")

In [ ]:
# ── 4. Build the two specialist nodes ─────────────────────────────────────────
# Key pattern: each node receives the FULL state and returns a DICT of
# only the fields it updates. LangGraph merges the dict into state.
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
_search = DuckDuckGoSearchResults(max_results=5)


def researcher_node(state: AgentState) -> dict:
    p = state["prospect"]
    query = f"{p.first_name} {p.last_name} {p.company} {p.position}"
    print(f"  [researcher] searching: {query}")
    raw = _search.run(query)

    resp = _llm.invoke([
        SystemMessage("Extract 2-3 factual, verifiable sentences about this person and company from the search results. Be concise."),
        HumanMessage(f"Prospect: {p.first_name} {p.last_name}, {p.position} at {p.company}\n\nSearch results:\n{raw[:2000]}"),
    ])
    return {"research": ResearchOutput(search_query=query, search_results=resp.content)}


def copywriter_node(state: AgentState) -> dict:
    p = state["prospect"]
    facts = state["research"].search_results
    print(f"  [copywriter] drafting for {p.first_name} {p.last_name}")

    structured_llm = _llm.with_structured_output(OutreachOutput)
    result = structured_llm.invoke([
        SystemMessage(
            "You are an expert outreach copywriter. Write a friendly, specific 1-sentence message "
            "(<=300 chars) that references a concrete fact about the prospect or their company. "
            "Do NOT pitch any services. Aim to start a genuine conversation."
        ),
        HumanMessage(f"Prospect: {p.first_name} {p.last_name}, {p.position} at {p.company}\n\nResearch facts:\n{facts}"),
    ])
    return {"output": result}


print("Nodes defined: researcher_node, copywriter_node")

In [ ]:
# ── 5. Wire the linear graph and run ──────────────────────────────────────────
# Linear pipeline: START -> researcher -> copywriter -> END
# No conditional edges needed — the flow is always the same.
from langgraph.graph import END, START, StateGraph

graph = StateGraph(AgentState)
graph.add_node("researcher", researcher_node)
graph.add_node("copywriter", copywriter_node)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "copywriter")
graph.add_edge("copywriter", END)
app = graph.compile()

# Use a well-known public figure so DuckDuckGo will find real facts
PROSPECT = ProspectMetadata(
    first_name="Jensen",
    last_name="Huang",
    company="NVIDIA",
    position="CEO",
)

print(f"Running pipeline for: {PROSPECT.first_name} {PROSPECT.last_name}, {PROSPECT.position} at {PROSPECT.company}\n")
result = app.invoke({"prospect": PROSPECT, "research": None, "output": None})

print("\n--- Research facts ---")
print(result["research"].search_results)

print("\n--- Generated outreach message ---")
print(result["output"].generated_message)
print(f"Confidence: {result['output'].confidence:.2f}")

## Exercises

**Exercise 1 — Change the prospect:** Replace `PROSPECT` with someone from your own network. Does the researcher find useful facts? Does the outreach message feel specific and relevant?

**Exercise 2 — Add a third node:** Add a `qa_node` that checks `state['output'].generated_message` is <= 300 chars. If too long, print a warning. Wire it: `copywriter → qa → END`.

**Exercise 3 — Separate LLMs:** The original script uses a deep-research model for the researcher and a faster model for the copywriter. Create two separate `ChatOpenAI` instances with different temperatures. Does the message quality change?

**Exercise 4 — Batch mode:** Wrap `app.invoke()` in a loop over a list of 3 different prospects. Print a table of names and generated messages.

## What's next

- **6-multi-agent-graph** — three specialist agents sharing one state graph
- **13-structured-output** — deeper use of `with_structured_output` with complex schemas
- **8-new-idea-gen** — LLM generates and critiques its own ideas in a loop